# KG1 v141 — Nemotron-3-Nano-30B SFT (Colab Pro+ A100/H100)

## Objetivo: TARGET 0.92 (consensus 28 APIs)

### Datasets (~11,402 CoTs total):
- **Tong corpus** — 6,014 (`felipesp1983/nemotron-cot-tong-dgxchen`)
- **Cryptarithm 813** — 813 (`felipesp1983/nemotron-cryptarithm-cot-813`)
- **EqGuess 150** — 150 (DeepSeek-R1 distill via gist)
- **Synth Z3 4425** — 4,425 (`felipesp1983/nemotron-cryptarithm-synth-4425`)

### Hyperparams (consenso triple check):
- LoRA r=32 alpha=32 all-linear+lm_head
- **lr=5e-5** (consenso 23 APIs)
- **epochs=2** (consenso 18 APIs) com cosine scheduler
- max_length=8192 (Kaggle limit)
- batch effective 32 (grad_accum=32, micro=1)
- Chat template enable_thinking=True
- Prompt masking labels=-100
- Curriculum: tong → eq_guess → cryptarithm_813 → synth_4425

### Setup obrigatório:
1. **Runtime → Change runtime type → A100 (HighRAM)** ou H100
2. **Colab Secrets** (🔑 ícone lateral esquerdo):
   - `HF_KEY` (Write+Jobs perms - usado SOMENTE no Cell 9)
   - `KAGGLE_USERNAME`
   - `KAGGLE_KEY`
3. **Runtime → Run all**

Tempo estimado: **3-6 horas** (dependendo do GPU)


In [ ]:
# CELL 1: Setup secrets + GPU check
import os, sys

try:
    from google.colab import userdata
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print('[WARN] Not in Colab')

# HF token: HF_KEY preferred, HF_TOKEN fallback
if IS_COLAB:
    hf_token = None
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                print(f'HF token set via secret {key_name} (len={len(v)})')
                break
        except Exception:
            pass
    if not hf_token:
        print('[WARN] HF_KEY/HF_TOKEN not in Colab Secrets - configure before Cell 9')

    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'Kaggle: {os.environ["KAGGLE_USERNAME"]}')
    except Exception:
        print('[INFO] Kaggle creds not set (optional for training)')

# GPU
import torch
print()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')

import psutil
print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB')


In [ ]:
# CELL 2: Install dependencies (Colab has CUDA + PyTorch preinstalled)
%pip install -q --upgrade transformers>=4.48.0 peft>=0.14.0 trl>=0.14.0 datasets>=3.0.0 accelerate>=1.3.0 bitsandbytes>=0.45.0 huggingface_hub>=0.27.0
%pip install -q einops packaging ninja triton

import transformers, peft, trl
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'trl: {trl.__version__}')


In [ ]:
# CELL 3: Install mamba-ssm v2.3.1 + causal-conv1d v1.6.1.post4 (Colab Python 3.12 + Torch 2.10)
import sys, torch, subprocess

py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'  # e.g. cp312
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # e.g. 2.10
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'

# CUDA major: torch.version.cuda = '12.8' or '12.6' -> cu12 wheels work for both
cuda_full = torch.version.cuda or '12.8'
cuda_major = 'cu' + cuda_full.split('.')[0] + cuda_full.split('.')[1][0] if cuda_full else 'cu12'
# But mamba wheels use simply 'cu12' for any cuda 12.x
cuda_str = 'cu12'
print(f'Python: {py_ver} | Torch: {torch_short} | CUDA full: {cuda_full} | wheel uses: {cuda_str} | ABI: {abi_str}')

# v2.3.1 supports torch 2.8/2.9/2.10/2.11 with cp310-cp313 (Mar 2026)
# v1.6.1.post4 of causal-conv1d supports same combos
MAMBA_VER = '2.3.1'
CC_TAG = '1.6.1.post4'  # release tag
CC_ASSET_VER = '1.6.1'  # asset filename version (sem post4)

def install_combo(cu, t, abi):
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{CC_TAG}/causal_conv1d-{CC_ASSET_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VER}/mamba_ssm-{MAMBA_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    for url in [cc_url, mb_url]:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', url],
                          capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            return False, r.stderr[-200:]
    return True, ''

# Order: detected combo FIRST, then fallbacks (same major torch version)
attempts = [
    (cuda_str, torch_short, abi_str),
]
# Add ABI fallback (TRUE/FALSE)
opposite_abi = 'FALSE' if abi_str == 'TRUE' else 'TRUE'
attempts.append((cuda_str, torch_short, opposite_abi))
# Add neighboring torch versions
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        combo = (cuda_str, t, abi)
        if combo not in attempts:
            attempts.append(combo)

success = False
last_err = ''
for cu, t, abi in attempts:
    print(f'Trying: {cu} torch{t} abi{abi}...')
    ok, err = install_combo(cu, t, abi)
    if ok:
        print(f'  [OK] Installed: mamba_ssm {MAMBA_VER} + causal_conv1d {CC_VER} ({cu} torch{t} abi{abi})')
        success = True
        break
    else:
        last_err = err
        # Show short error message
        err_short = err.replace('\n', ' ')[:150]
        print(f'  [FAIL] {err_short}')

if not success:
    print()
    print('Wheels failed. Last error:', last_err[:300])
    print('Fallback: install via pip (will try build from source)...')
    import os
    os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
    os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--force-reinstall',
                    f'causal-conv1d=={CC_ASSET_VER}'], timeout=1200)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--force-reinstall',
                    f'mamba-ssm=={MAMBA_VER}'], timeout=1500)

# Force reload of modules in case of stale cache
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    if m in sys.modules:
        del sys.modules[m]

# Verify imports
print()
print('=== Final verification ===')
all_ok = True
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    try:
        mod = __import__(m)
        v = getattr(mod, '__version__', 'imported')
        print(f'  [OK] {m}: {v}')
    except ImportError as e:
        print(f'  [FAIL] {m}: {e}')
        all_ok = False

if not all_ok:
    print()
    print('!!! Some modules failed - cannot proceed to Cell 6 (model load)')
    print('Try: Runtime -> Restart runtime, then re-run Cells 1-3')
else:
    print()
    print('SUCCESS: All mamba-ssm CUDA kernels available!')


In [ ]:
# CELL 4: Download all 4 datasets via direct HTTP (no token needed)
import os, urllib.request

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}

for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    print(f'Downloading {name}...')
    urllib.request.urlretrieve(url, out)
    sz = os.path.getsize(out)
    if sz > 1e6:
        print(f'  [OK] {name}: {sz/1e6:.2f} MB')
    else:
        print(f'  [OK] {name}: {sz/1e3:.1f} KB')

tong_path = os.path.join(DATA_DIR, 'tong.csv')
crypt_path = os.path.join(DATA_DIR, 'cryptarithm_813.jsonl')
eq_guess_path = os.path.join(DATA_DIR, 'eq_guess_150.jsonl')
synth_path = os.path.join(DATA_DIR, 'synth_4425.jsonl')


In [ ]:
# CELL 5: Format + Merge + Curriculum (easy -> hard)
import csv, json
from datasets import Dataset

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

all_data = []

# 1. Tong corpus (6014)
n_tong = 0
with open(tong_path, encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': cot,
                'source': 'tong',
            })
            n_tong += 1

# 2. Cryptarithm 813
n_crypt = 0
with open(crypt_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid') and row.get('cot'):
            all_data.append({
                'prompt': row['prompt'].strip() + PROMPT_SUFFIX,
                'completion': row['cot'].strip(),
                'source': 'cryptarithm_813',
            })
            n_crypt += 1

# 3. EqGuess 150
n_eq = 0
with open(eq_guess_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': cot,
                'source': 'eq_guess_150',
            })
            n_eq += 1

# 4. Synth Z3 4425
n_synth = 0
with open(synth_path, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({
                'prompt': prompt + PROMPT_SUFFIX,
                'completion': cot,
                'source': 'synth_4425',
            })
            n_synth += 1

print(f'Tong:               {n_tong}')
print(f'Cryptarithm 813:    {n_crypt}')
print(f'EqGuess 150:        {n_eq}')
print(f'Synth Z3 4425:      {n_synth}')
print(f'TOTAL:              {len(all_data)}')

# Curriculum: easy first (tong already covers most families) -> hard (cryptarithm + synth)
SOURCE_ORDER = {'tong': 0, 'eq_guess_150': 1, 'cryptarithm_813': 2, 'synth_4425': 3}
all_data.sort(key=lambda x: SOURCE_ORDER.get(x['source'], 99))

ds = Dataset.from_list(all_data)
print()
print('Curriculum order: tong -> eq_guess -> cryptarithm_813 -> synth_4425')


In [ ]:
# CELL 6: Load Nemotron-3-Nano-30B + LoRA (auto NF4 if VRAM < 70GB)
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_NF4 = vram_gb < 70  # H100 80GB / A100 80GB = no NF4 needed
print(f'VRAM: {vram_gb:.1f} GB -> USE_NF4: {USE_NF4}')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, token=os.environ.get('HF_TOKEN')
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.time()
if USE_NF4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={'': 0},
        trust_remote_code=True,
        token=os.environ.get('HF_TOKEN'),
        attn_implementation='eager',
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
        token=os.environ.get('HF_TOKEN'),
        attn_implementation='eager',
    )
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()

print(f'[OK] Model loaded in {time.time()-t0:.1f}s')
print(f'VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=TARGET_REGEX,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'[OK] LoRA trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B')
print(f'VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB total')


In [ ]:
# CELL 7: Tokenize com chat template + prompt masking
MAX_LENGTH = 8192


def fmt(ex):
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        enable_thinking=True,
    )
    if len(full_ids) > MAX_LENGTH:
        full_ids = full_ids[:MAX_LENGTH]

    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100

    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }


tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'[OK] Tokenized: {len(tokenized)} samples')


In [ ]:
# CELL 8: Train (lr=5e-5, epochs=2 - consenso triple check)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = '/content/output_v141'
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    optim='adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    remove_unused_columns=False,
    report_to='none',
    dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors='pt',
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

import time
t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print(f'[OK] Training in {train_time_min:.1f} min')


In [ ]:
# CELL 9: Save adapter + Upload to HF
from huggingface_hub import HfApi, create_repo

ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('Saved adapter files:')
for f in os.listdir(ADAPTER_DIR):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f}: {sz/1e6:.2f} MB')

DEST_REPO = 'felipesp1983/kg1-v141-final'

# Verify token still valid
hf_token_env = os.environ.get('HF_TOKEN')
if not hf_token_env:
    print('[ERR] HF_TOKEN not set! Adapter saved locally only.')
else:
    try:
        api = HfApi(token=hf_token_env)
        # Test token
        whoami = api.whoami()
        print(f'HF user: {whoami["name"]}')
        create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token_env)
        api.upload_folder(
            folder_path=ADAPTER_DIR,
            repo_id=DEST_REPO,
            repo_type='model',
            commit_message=f'v141 trained {train_time_min:.1f}min Colab (Tong+813+150+4425synth lr5e-5 ep2)',
        )
        print(f'[OK] Uploaded: https://huggingface.co/{DEST_REPO}')
    except Exception as e:
        print(f'[ERR] Upload failed: {e}')
        print(f'Generate fresh HF token at https://huggingface.co/settings/tokens')
        print(f'Update Colab Secret HF_KEY and re-run THIS cell')


In [ ]:
# CELL 10: GDrive backup (sempre roda, mesmo se HF upload falhar)
from google.colab import drive
drive.mount('/content/drive')

import shutil
GDRIVE_BACKUP = '/content/drive/My Drive/KG1_v141_adapter'
if os.path.exists(GDRIVE_BACKUP):
    shutil.rmtree(GDRIVE_BACKUP)
shutil.copytree(ADAPTER_DIR, GDRIVE_BACKUP)
print(f'[OK] Backup: {GDRIVE_BACKUP}')

import subprocess
sz = subprocess.run(['du', '-sh', GDRIVE_BACKUP], capture_output=True, text=True).stdout
print(f'Size: {sz.strip()}')

# Listar conteudo
print('Files:')
for f in os.listdir(GDRIVE_BACKUP):
    p = os.path.join(GDRIVE_BACKUP, f)
    sz = os.path.getsize(p)
    print(f'  {f}: {sz/1e6:.2f} MB')


## Próximos passos após v141 treinar

### 1. Quick eval (no Colab)
```python
# Test adapter generates correct \boxed{} format
prompt = "In Alice's Wonderland, ... Now, determine the result for: 58*47"
# ... eval script com 5-10 samples
```

### 2. Submit to Kaggle (terminal local)
```bash
python scripts/submit_kaggle.py \
    --hf-repo felipesp1983/kg1-v141-final \
    --reference-adapter-dir runs/attached_085_audit_20260416 \
    --test-csv data/kaggle/unzipped/test.csv \
    --message "v141 11400CoTs Tong+synth4425 lr5e-5 ep2" \
    --max-daily-submits 5
```

### 3. Score esperado (consenso 28 APIs)
- Mediana: **0.92**
- Range: [0.88 - 0.95]
- P(>= 0.87): 75-85%

### 4. Se score < 0.87
Próximo: **v142 GRPO** sobre v141 SFT
- Adaptar `rewards.py` do andy279 (open-r1 fork)
- Verifiable reward via Z3 (cryptarithm) + sympy (eq_numeric)
- +10% adicional esperado
